In [35]:

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [36]:
# CHARGEMENT DES DONNÉES
df = pd.read_csv("../01_Data/get_around_pricing_project.csv")
df = df.drop(columns=["Unnamed: 0"])

print(f"Shape : {df.shape}")
print(df.head())

Shape : (4843, 14)
  model_key  mileage  engine_power    fuel paint_color     car_type  \
0   Citroën   140411           100  diesel       black  convertible   
1   Citroën    13929           317  petrol        grey  convertible   
2   Citroën   183297           120  diesel       white  convertible   
3   Citroën   128035           135  diesel         red  convertible   
4   Citroën    97097           160  diesel      silver  convertible   

   private_parking_available  has_gps  has_air_conditioning  automatic_car  \
0                       True     True                 False          False   
1                       True     True                 False          False   
2                      False    False                 False          False   
3                       True     True                 False          False   
4                       True     True                 False          False   

   has_getaround_connect  has_speed_regulator  winter_tires  \
0                   Tr

In [37]:
#Features and target

TARGET    = "rental_price_per_day"
NUM_COLS  = ["mileage", "engine_power"]
CAT_COLS  = ["model_key", "fuel", "paint_color", "car_type"]
BOOL_COLS = [
    "private_parking_available", "has_gps", "has_air_conditioning",
    "automatic_car", "has_getaround_connect", "has_speed_regulator", "winter_tires",
]

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTrain : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")


Train : 3874 lignes | Test : 969 lignes


In [38]:
#Préprocessing
preprocessor = ColumnTransformer([
    ("num",  StandardScaler(),                                            NUM_COLS),
    ("cat",  OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
    ("bool", "passthrough",                                               BOOL_COLS),
])

In [39]:
# Mlflow
EXPERIMENT_NAME = "getaround-pricing"
mlflow.set_experiment(EXPERIMENT_NAME)

# Dictionnaire pour garder trace des runs et de leurs métriques
all_runs = {}

In [40]:
# RANDOM FOREST
with mlflow.start_run(run_name="RandomForest"):

    mlflow.set_tag("model_type", "RandomForest")

    # Paramètres
    n_estimators_rf = 100
    mlflow.log_param("n_estimators", n_estimators_rf)
    mlflow.log_param("random_state", 42)

    # Pipeline + entraînement
    pipe_rf = Pipeline([
        ("prep",  preprocessor),
        ("model", RandomForestRegressor(n_estimators=n_estimators_rf, random_state=42, n_jobs=-1)),
    ])
    pipe_rf.fit(X_train, y_train)
    preds_rf = pipe_rf.predict(X_test)

    # Métriques
    mae_rf  = round(mean_absolute_error(y_test, preds_rf), 4)
    rmse_rf = round(np.sqrt(mean_squared_error(y_test, preds_rf)), 4)
    r2_rf   = round(r2_score(y_test, preds_rf), 4)

    mlflow.log_metric("MAE",  mae_rf)
    mlflow.log_metric("RMSE", rmse_rf)
    mlflow.log_metric("R2",   r2_rf)

    print(f"[RandomForest]  MAE={mae_rf}  RMSE={rmse_rf}  R²={r2_rf}")

    mlflow.sklearn.log_model(pipe_rf, artifact_path="model", input_example=X_test.iloc[:2])

    run_id_rf = mlflow.active_run().info.run_id

all_runs["RandomForest"] = {"run_id": run_id_rf, "R2": r2_rf, "pipeline": pipe_rf}

[RandomForest]  MAE=10.679  RMSE=16.7348  R²=0.7341


/opt/anaconda3/envs/ml/lib/python3.10/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [41]:
# Gradient Boosting

with mlflow.start_run(run_name="GradientBoosting"):

    mlflow.set_tag("model_type", "GradientBoosting")

    # Paramètres
    n_estimators_gb = 200
    learning_rate   = 0.1
    max_depth       = 3
    mlflow.log_param("n_estimators",  n_estimators_gb)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("max_depth",     max_depth)
    mlflow.log_param("random_state",  42)

    # Pipeline + entraînement
    pipe_gb = Pipeline([
        ("prep",  preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=n_estimators_gb,
            learning_rate=learning_rate,
            max_depth=max_depth,
            random_state=42,
        )),
    ])
    pipe_gb.fit(X_train, y_train)
    preds_gb = pipe_gb.predict(X_test)

    # Métriques
    mae_gb  = round(mean_absolute_error(y_test, preds_gb), 4)
    rmse_gb = round(np.sqrt(mean_squared_error(y_test, preds_gb)), 4)
    r2_gb   = round(r2_score(y_test, preds_gb), 4)

    mlflow.log_metric("MAE",  mae_gb)
    mlflow.log_metric("RMSE", rmse_gb)
    mlflow.log_metric("R2",   r2_gb)

    print(f"[GradientBoosting]  MAE={mae_gb}  RMSE={rmse_gb}  R²={r2_gb}")

    mlflow.sklearn.log_model(pipe_gb, artifact_path="model", input_example=X_test.iloc[:2])

    run_id_gb = mlflow.active_run().info.run_id

all_runs["GradientBoosting"] = {"run_id": run_id_gb, "R2": r2_gb, "pipeline": pipe_gb}

[GradientBoosting]  MAE=10.9054  RMSE=16.4099  R²=0.7443


/opt/anaconda3/envs/ml/lib/python3.10/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [42]:
# Comparaison and saving the best model

print("\n--- Récapitulatif ---")
for name, info in all_runs.items():
    print(f"  {name:35s}  R²={info['R2']:.4f}")

best_name = max(all_runs, key=lambda k: all_runs[k]["R2"])
best_pipeline = all_runs[best_name]["pipeline"]
best_run_id   = all_runs[best_name]["run_id"]

print(f"\nMeilleur modèle : {best_name}  (R²={all_runs[best_name]['R2']:.4f})")

joblib.dump(best_pipeline, "../04_Deployement/API/model.pkl")
print(f"Modèle sauvegardé")
print(f"Run MLflow → {best_run_id}")




--- Récapitulatif ---
  RandomForest                         R²=0.7341
  GradientBoosting                     R²=0.7443

Meilleur modèle : GradientBoosting  (R²=0.7443)
Modèle sauvegardé
Run MLflow → 9afc2012442640bcb9b69144b417554b
